In [ ]:
import os

os.chdir("..")
os.getcwd()

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image

from utils.dofa_clip.factory import create_model_from_pretrained, get_tokenizer

In [ ]:
# Load the pre-trained DOFA-CLIP model
hf_repo = "hf-hub:earthflow/GeoLB-ViT-14-SigLIP-so400m-384-EO"
model, preprocess = create_model_from_pretrained(
    hf_repo, cache_dir="/Users/gabriele/.cache/huggingface/hub/"
)
tokenizer = get_tokenizer(hf_repo)

In [ ]:
# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "mps"
model = model.to(device)

# Load and preprocess your image
image = Image.open("airplane.png")
image

In [ ]:
# Define your text labels
labels_list = [
    "A busy airport with many aeroplanes.",
    "Satellite view of Hohai university.",
    "Satellite view of sydney",
    "Many people in a stadium",
]

# Tokenize text
text = tokenizer(labels_list, context_length=model.context_length)
text = text.to(device)

In [ ]:
# Perform inference
with torch.no_grad():
    # Define wavelengths for your specific modality
    # These are example wavelengths for RGB bands
    wvs = torch.tensor([0.665, 0.560, 0.490]).to(device)

    # Encode image and text
    image_features = model.visual.trunk(image, wvs)[0]
    text_features = model.encode_text(text)

    # Normalize features
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    # Calculate probabilities
    text_probs = torch.sigmoid(
        image_features @ text_features.T * model.logit_scale.exp() + model.logit_bias
    )

# Display results
zipped_list = list(zip(labels_list, [round(p.item(), 3) for p in text_probs[0]]))
print("Label probabilities:", zipped_list)